# Task 1.2 — Key Assumptions
**Paper:** *Scalable Training of Mixture Models via Coresets* (Feldman, Faulkner, Krause — NIPS 2011)

---

## Assumption 1

**Assumption:** The covariance matrices of each Gaussian component are *ε-semi-spherical*, meaning all eigenvalues lie in the bounded interval [ε, 1/ε] for some small ε > 0.

**Why the method needs it:** The core theoretical reduction (Lemma 3.2) maps the GMM log-likelihood into a geometric problem of approximating distances between points and subspaces in a transformed 2d-dimensional space. For this reduction to yield a *k-means*-style problem (with subspaces collapsing to points), the Gaussians must not be degenerate — eigenvalues bounded away from 0 and infinity ensure the covariance ellipsoids are neither infinitely flat (rank-deficient) nor infinitely round, so level sets remain well-behaved ellipses. Without this, the multiplicative approximation error blows up.

**Violation scenario:** A dataset with a highly anisotropic cluster — for example, measurements of a long thin river valley where one spatial dimension has variance ~10,000 and the other ~0.001 — would violate this assumption, because one eigenvalue would be near zero and another very large, taking the ratio far outside [ε, 1/ε].

**Paper reference:** Section 2 (definition of constraint set C_ε), Section 3.1 (the reduction to k-means requires semi-spherical assumption, stated explicitly: *"for semi-spherical Gaussians, it can be shown that the subspaces can be chosen as points while incurring a multiplicative error of at most 1/ε"*), and Section 4 (where relaxing this assumption is discussed as a generalisation).

---

## Assumption 2

**Assumption:** The rough initial clustering B, built by the iterative uniform-sampling-and-halving procedure, provides a *sufficiently accurate geometric cover* of all clusters in the data — meaning no major cluster is entirely missed or severely under-represented in B.

**Why the method needs it:** The importance sampling probabilities m(x) are defined relative to B (the 5/|D_b| cluster-coverage term and the dist(x, B)² term). If B misses a cluster entirely — leaving a region of the data far from any point in B — then every point in that cluster gets a high distance-based probability and a low weight γ, but the *cluster-membership* correction (5/|D_b|) doesn't activate. The theoretical proof of Theorem 3.1 relies on B being a constant-factor approximation to the k-means optimum, which implicitly requires coverage of all clusters.

**Violation scenario:** A dataset with one extremely small cluster (say, 5 out of 10,000 points) that is geometrically close to a much larger cluster — the halving procedure would almost certainly eliminate all 5 points early in the while-loop because they are near the larger cluster's sampled points, so B would not contain any representative of this tiny cluster.

**Paper reference:** Section 3 (Algorithm 1, the while-loop building B), and the discussion in Section 3.1 that B must be replaced by a constant-factor k-means approximation B' to get a coreset size independent of n — this replacement is necessary precisely because naive B may not cover all clusters well.

---

## Assumption 3

**Assumption:** The number of mixture components k is known and fixed in advance; the method does not perform model selection over k.

**Why the method needs it:** The coreset size bound O(dk³/ε²) depends explicitly on k, and both Algorithm 1 (β = 10dk ln(1/δ)) and Algorithm 2 (Weighted EM initialised with k components) treat k as a hard-coded hyperparameter. The theoretical guarantee in Theorem 3.1 is a uniform bound over all θ ∈ C_ε, where C_ε is defined as mixtures of *exactly k* Gaussians. If k is wrong (e.g., the true data has 8 clusters but k=3 is used), the coreset is still valid for that k, but the fitted model will be poor — and the paper provides no mechanism to detect or correct this.

**Violation scenario:** A customer transaction dataset where the number of distinct purchasing behaviours is unknown and might range from 5 to 50 — choosing k incorrectly means the coreset is optimised for the wrong complexity class, and standard coreset-based EM cannot adaptively discover the true k.

**Paper reference:** Section 2 (problem statement explicitly parameterised by k), Algorithm 1 (β = 10dk ln(1/δ) uses k), Theorem 3.1 (bound holds *for all θ ∈ C_ε*, where C_ε fixes k), and Section 5 (all experiments fix k: k=10 for MNIST, k=33 for tetrode, k=6 for CSN).

---

## Assumption 4 (Bonus)

**Assumption:** Data points are processed as *exchangeable real-valued vectors* — the method assumes Euclidean distance is a meaningful measure of similarity, and the geometric reduction (Lemma 3.2) relies on the Euclidean structure of R^d.

**Why the method needs it:** The entire coreset construction uses dist(x, B) = Euclidean distance, and the reduction to k-means in the transformed space φ(x) ∈ R^{2d} relies on the fact that Gaussian level sets are ellipses in Euclidean space. The soft-min approximation and the triangle inequality bound in the proof both use properties of the Euclidean norm.

**Violation scenario:** Text documents represented as bag-of-words vectors — cosine similarity, not Euclidean distance, is the natural metric, and applying Euclidean-based sampling probabilities would distort the importance weights, potentially over-sampling documents with long vectors (many words) regardless of their semantic representativeness.

**Paper reference:** Lemma 3.2 (the key reduction maps x ∈ R^d to φ(x) ∈ R^{2d} using Euclidean distances to subspaces), Section 4 extension to ℓ_q distances which is presented as a non-trivial generalisation requiring a new Hölder inequality argument.